In [1]:
import requests
from bs4 import BeautifulSoup

In [2]:
url = 'https://news.google.com/rss/headlines/section/topic/BUSINESS'
response = requests.get(url)
soup = BeautifulSoup(response.content, 'xml')

In [3]:
headlines = soup.find('channel').find_all('title')

In [4]:
hlist = []
headlines = soup.find('channel').find_all('title')
for item in headlines:
    hlist.append(item.get_text())
del hlist[0:2]

In [5]:
!pip install --quiet torch
import torch
print(torch.__version__)

2.9.0+cpu


In [6]:
!pip install --quiet transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

In [7]:
import warnings
warnings.filterwarnings('ignore')
source_model = f"ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(source_model)
model = AutoModelForSequenceClassification.from_pretrained(source_model)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
example_text = hlist[1]
example_text

'Man charged in arson attack on Sam Altman’s house had AI CEO kill list, prosecutors say - Fortune'

In [9]:
encoded_text = tokenizer(example_text, return_tensors = 'pt')
encoded_text

{'input_ids': tensor([[  101,  2158,  5338,  1999, 24912,  2886,  2006,  3520, 12456,  2386,
          1521,  1055,  2160,  2018,  9932,  5766,  3102,  2862,  1010, 19608,
          2360,  1011,  7280,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [10]:
output = model(**encoded_text)
output

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

SequenceClassifierOutput(loss=None, logits=tensor([[-1.5888,  1.0915,  1.5982]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [11]:
scores = output[0][0].detach().numpy()
scores = softmax(scores)
scores

array([0.02512197, 0.36651504, 0.60836303], dtype=float32)

In [19]:
def get_sentiment(example):
    encoded_text = tokenizer(example, return_tensors = 'pt')
    output = model(**encoded_text)
    scores = output[0][0].detach().numpy()
    scores = softmax(scores)
    scores_dict = {
        'finbert_pos' : scores[0],
        'finbert_neg' : scores[1],
        'finbert_neu' : scores[2]
    }
    return scores_dict

In [20]:
get_sentiment(hlist[1])

{'finbert_pos': 0.025121974,
 'finbert_neg': 0.36651504,
 'finbert_neu': 0.60836303}

In [21]:
import pandas as pd
headlines_df = pd.DataFrame(hlist, columns = ['Headlines'])

In [22]:
hl_sentiment = headlines_df['Headlines'].apply(get_sentiment)

In [23]:
finbert_neg = []
for row in hl_sentiment:
    finbert_neg.append(list(row.values())[0])
finbert_neut = []
for row in hl_sentiment:
    finbert_neut.append(list(row.values())[1])
finbert_pos = []
for row in hl_sentiment:
    finbert_pos.append(list(row.values())[2])

In [24]:
final_hl_df = pd.DataFrame({
    'Headline': hlist,
    'Positive finBERT Score': finbert_neg,
    'Negative finBERT Score': finbert_neut,
    'Neutral finBERT Score': finbert_pos
})
    

In [25]:
final_hl_df

,Headline,Positive finBERT Score,Negative finBERT Score,Neutral finBERT Score
0,Middle East War Will Slow Global Economic Grow...,0.013938,0.891151,0.094911
1,Man charged in arson attack on Sam Altman’s ho...,0.025122,0.366515,0.608363
2,JPMorgan profit beats estimates on record trad...,0.849358,0.046519,0.104123
3,US wholesale prices surged 4% last month after...,0.849473,0.038681,0.111847
4,"Lucid Motors names new CEO, lands more money f...",0.433377,0.018366,0.548256
...,...,...,...,...
62,Mercedes-Benz unveils new EQS with 926 km rang...,0.472656,0.008871,0.518473
63,"Why fresh tomato prices have skyrocketed, how ...",0.060691,0.043465,0.895844
64,Why two Wall Street titans just turned bullish...,0.047533,0.173902,0.778566
65,Oil supply crunch intensifies as last Hormuz t...,0.047620,0.291642,0.660738
